# Train DACMF-DTI on Google Colab (Free T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-kumar-277/Generalized-EnsDTI/blob/main/notebooks/train_dacmf_colab.ipynb)

This notebook trains the **DACMF-DTI** deep learning expert model (`AttentionDTA`) across all four leakage-controlled evaluation regimes on the Davis dataset:
1. `random_pair` (standard interaction baseline)
2. `cold_drug` (unseen drug candidates)
3. `cold_protein` (unseen protein targets)
4. `double_cold` (both drug and target unseen)

**Hardware requirement**: Select **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not active. Please select Runtime > Change runtime type > T4 GPU")

In [ ]:
# 2. Install dependencies
!pip install -q rdkit pandas numpy scikit-learn

In [ ]:
# 3. Clone Generalized-EnsDTI repository
import os
if not os.path.exists("Generalized-EnsDTI"):
    !git clone https://github.com/sai-kumar-277/Generalized-EnsDTI.git
%cd Generalized-EnsDTI
!git pull origin main
!ls -la

In [ ]:
# 4. Optional: Mount Google Drive to save predictions permanently
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive successfully mounted.")
except Exception as e:
    DRIVE_MOUNTED = False
    print("Drive mount skipped or unavailable. Output will be zipped for direct browser download.")

In [ ]:
# 5. Run GPU Training for DACMF-DTI across all 4 split regimes
# Each epoch takes ~40-50s on T4 GPU (approx 10 minutes total for 5 epochs x 4 splits)
!python train_dacmf_colab.py --split all --epochs 5 --batch-size 64 --lr 1e-4

In [ ]:
# 6. Verify generated prediction files
import glob, pandas as pd
pred_files = glob.glob("tmp/davis/*/dacmf_predictions.csv")
print(f"Found {len(pred_files)} prediction files:")
for f in sorted(pred_files):
    df = pd.read_csv(f)
    test_df = df[df["split"] == "test"]
    print(f"  {f} | Total: {len(df)} rows | Test: {len(test_df)} rows | Mean prob: {df['predicted_probability'].mean():.4f}")

In [ ]:
# 7. Package predictions into ZIP and download to local machine
!zip -r dacmf_predictions.zip tmp/davis/*/dacmf_predictions*.csv

# Save to Google Drive if mounted
if DRIVE_MOUNTED:
    !mkdir -p "/content/drive/MyDrive/Generalized-EnsDTI/tmp/davis"
    !cp -r tmp/davis/* "/content/drive/MyDrive/Generalized-EnsDTI/tmp/davis/"
    print("Successfully saved prediction files to Google Drive: MyDrive/Generalized-EnsDTI/tmp/davis/")

# Direct download trigger in Colab browser
try:
    from google.colab import files
    files.download("dacmf_predictions.zip")
    print("Download prompt triggered for dacmf_predictions.zip!")
except Exception as e:
    print(f"Browser download prompt not available ({e}). Download dacmf_predictions.zip from the Colab file tree.")